In [11]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from copy import deepcopy
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
jobs_raw_df = pd.read_csv("../processed_data/data_isolated_agg.csv", index_col=0)
jobs_raw_df = jobs_raw_df[jobs_raw_df["Num Nodes"] == 8]
inhibitors_raw_df = pd.read_csv("../processed_data/data_inhib_isolated_agg.csv", index_col=0)
inhibitors_raw_df = inhibitors_raw_df[inhibitors_raw_df["Num Nodes"] == 8]
job_inh_raw_df = pd.read_csv("../processed_data/data_inhib_coscheduled_agg.csv", index_col=0)
job_inh_raw_df = job_inh_raw_df[job_inh_raw_df["Num Nodes"] == 8]
pair_raw_df = pd.read_csv("../processed_data/data_coscheduled_agg.csv", index_col=0)
pair_raw_df = pair_raw_df[pair_raw_df["Num Nodes"] == 8]

In [2]:
# Preprocess data #1

# Isolated jobs
jobs_df = pd.DataFrame()
jobs_df["job_id"] = jobs_raw_df["App"]
jobs_df["mpi_time"] = jobs_raw_df["MPI Time Average Mean"]
jobs_df["comm_frac"] = jobs_raw_df["MPI Time Average Mean"] / jobs_raw_df["App Time Average Mean"]
jobs_df["total_msgs"] = jobs_raw_df["Total Messages Sent Mean"]
jobs_df["total_bytes"] = jobs_raw_df["Total Bytes Sent Mean"]

# Isolated inhibitors
inhibitors_df = pd.DataFrame()
columns_to_use = ["Num Nodes", "Inhib Message Size", "Inhib Wait Time (us)", "Inhib Comm Sparsity"]
selected_data = inhibitors_raw_df[columns_to_use]
inhibitors_df["inhib_id"] = (selected_data
                             .astype(str)
                             .agg('_'.join, axis=1))
inhibitors_df["msg_size"] = inhibitors_raw_df["Inhib Message Size"]
inhibitors_df["wait_time"] = inhibitors_raw_df["Inhib Wait Time (us)"]
inhibitors_df["comm_sparsity"] = inhibitors_raw_df["Inhib Comm Sparsity"]
inhibitors_df["mpi_time"] = inhibitors_raw_df["Inhib MPI Time Average Mean"]
inhibitors_df["comm_frac"] = inhibitors_raw_df["Inhib MPI Time Average Mean"] / inhibitors_raw_df["Inhib App Time Average Mean"]
inhibitors_df["total_msgs"] = inhibitors_raw_df["Inhib Total Messages Sent Mean"]
inhibitors_df["total_bytes"] = inhibitors_raw_df["Inhib Total Bytes Sent Mean"]

# Co-scheduled job and inhibitors
job_inh_df = pd.DataFrame()
job_inh_df["job_id"] = job_inh_raw_df["App"]
columns_to_use = ["Num Nodes", "Inhib Message Size", "Inhib Wait Time (us)", "Inhib Comm Sparsity"]
selected_data = job_inh_raw_df[columns_to_use]
job_inh_df["inhib_id"] = (selected_data
                             .astype(str)
                             .agg('_'.join, axis=1))
job_inh_slowdowns = []
for i in job_inh_raw_df.iloc:
    iso_time = (jobs_df[jobs_df["job_id"] == i["App"]])["mpi_time"].max()
    coscheduled_time = i["MPI Time Average Mean"]
    if (coscheduled_time < iso_time):
        slowdown = 1
    else:
        slowdown = coscheduled_time/iso_time
    job_inh_slowdowns.append(slowdown)
job_inh_df["slowdown"] = job_inh_slowdowns

# Co-scheduled jobs
pair_df = pd.DataFrame()
pair_df["jobA_id"] = pair_raw_df["App A"]
pair_df["jobB_id"] = pair_raw_df["App B"]
pair_slowdowns_a = []
pair_slowdowns_b = []
for i in pair_raw_df.iloc:
    iso_time_a = (jobs_df[jobs_df["job_id"] == i["App A"]])["mpi_time"].max()
    iso_time_b = (jobs_df[jobs_df["job_id"] == i["App B"]])["mpi_time"].max()
    coscheduled_time_a = i["App A MPI Time Average Mean"]
    coscheduled_time_b = i["App B MPI Time Average Mean"]
    if (coscheduled_time_a < iso_time_a):
        slowdown_a = 1
    else:
        slowdown_a = coscheduled_time_a/iso_time_a
    pair_slowdowns_a.append(slowdown_a)
    if (coscheduled_time_b < iso_time_b):
        slowdown_b = 1
    else:
        slowdown_b = coscheduled_time_b/iso_time_b
    pair_slowdowns_b.append(slowdown_b)
pair_df["slowdown_A"] = pair_slowdowns_a
pair_df["slowdown_B"] = pair_slowdowns_b

NameError: name 'jobs_raw_df' is not defined

In [6]:
jobs_df.to_csv("utils/few_shot/data/jobs.csv")
inhibitors_df.to_csv("utils/few_shot/data/inhibitors.csv")
job_inh_df.to_csv("utils/few_shot/data/job_inh.csv")
pair_df.to_csv("utils/few_shot/data/pair.csv")

In [3]:
# Few shot utils have the experimentation pipeline. Rest of this notebook analyzes the results from the pipeline

In [27]:
# Add times to prediction datasets

jobs_df = pd.read_csv("utils/few_shot/data/jobs.csv", index_col=0)
inhibitors_df = pd.read_csv("utils/few_shot/data/inhibitors.csv", index_col=0)
job_inh_df = pd.read_csv("utils/few_shot/data/job_inh.csv", index_col=0)
pair_df = pd.read_csv("utils/few_shot/data/pair.csv", index_col=0)

def append_times_to_preds(preds: pd.DataFrame):
    app_A_iso_times = []
    for i in preds.iloc:
        app_A = i["app_A"]
        app_B = i["app_B"]
        app_A_iso_time = jobs_df[jobs_df["job_id"] == app_A]["mpi_time"].max()
        app_A_iso_times.append(app_A_iso_time)
    preds["app_A_iso_time"] = app_A_iso_times
    preds["app_A_y_true_time"] = preds["app_A_iso_time"] * preds["y_true"]
    preds["app_A_y_pred_time"] = preds["app_A_iso_time"] * preds["y_pred"]

train_df = pd.read_csv("utils/few_shot/output/train.csv")
test_df= pd.read_csv("utils/few_shot/output/test.csv")

append_times_to_preds(train_df)
append_times_to_preds(test_df)
train_df

,app_A,app_B,set_A,set_B,b_A,b_B,b_A_raw,b_B_raw,y_true,y_pred,app_A_iso_time,app_A_y_true_time,app_A_y_pred_time
0,fiesta,minivite,"[[-0.5094369053840637, 0.8200529217720032, -0....","[[0.10417995601892471, 0.8200529217720032, -0....","[-0.023841898888349533, -0.043366845697164536,...","[-0.26013436913490295, -0.3027505874633789, -0...","[65.67544555664062, 0.18289333581924438, 64512...","[51.597267150878906, 0.13898253440856934, 2408...",1.051300,1.083773,65.675446,69.044584,71.177253
1,amg,minife,"[[0.6035680770874023, 0.8200529217720032, -0.5...","[[0.44503530859947205, 0.8200529217720032, -0....","[-0.9116873145103455, -0.9033464789390564, 0.0...","[-0.8978692889213562, -0.9167898297309875, -0....","[12.778085708618164, 0.03730827942490578, 1064...","[13.60135555267334, 0.035032469779253006, 6788...",1.000000,1.076284,12.778086,12.778086,13.752855
2,minife,fiesta,"[[0.44503530859947205, 0.8200529217720032, -0....","[[-0.5094369053840637, 0.8200529217720032, -0....","[-0.8978692889213562, -0.9167898297309875, -0....","[-0.023841898888349533, -0.043366845697164536,...","[13.60135555267334, 0.035032469779253006, 6788...","[65.67544555664062, 0.18289333581924438, 64512...",1.000000,1.078313,13.601356,13.601356,14.666512
3,beatnik,amg,"[[0.32986029982566833, 0.8200529217720032, -0....","[[0.6035680770874023, 0.8200529217720032, -0.5...","[0.5507001876831055, 0.36005765199661255, -0.3...","[-0.9116873145103455, -0.9033464789390564, 0.0...","[99.90636444091797, 0.2511886656284332, 199961...","[12.778085708618164, 0.03730827942490578, 1064...",1.144124,1.114912,99.906362,114.305245,111.386840
4,amg,fiesta,"[[0.6035680770874023, 0.8200529217720032, -0.5...","[[-0.5094369053840637, 0.8200529217720032, -0....","[-0.9116873145103455, -0.9033464789390564, 0.0...","[-0.023841898888349533, -0.043366845697164536,...","[12.778085708618164, 0.03730827942490578, 1064...","[65.67544555664062, 0.18289333581924438, 64512...",1.027639,1.086904,12.778086,13.131255,13.888549
...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,minivite,fiesta,"[[0.10417995601892471, 0.8200529217720032, -0....","[[-0.5094369053840637, 0.8200529217720032, -0....","[-0.26013436913490295, -0.3027505874633789, -0...","[-0.023841898888349533, -0.043366845697164536,...","[51.597267150878906, 0.13898253440856934, 2408...","[65.67544555664062, 0.18289333581924438, 64512...",1.134550,1.087682,51.597266,58.539676,56.121411
74,minife,beatnik,"[[0.44503530859947205, 0.8200529217720032, -0....","[[0.32986029982566833, 0.8200529217720032, -0....","[-0.8978692889213562, -0.9167898297309875, -0....","[0.5507001876831055, 0.36005765199661255, -0.3...","[13.60135555267334, 0.035032469779253006, 6788...","[99.90636444091797, 0.2511886656284332, 199961...",1.346966,1.210056,13.601356,18.320569,16.458397
75,lammps,minife,"[[-0.23122406005859375, 0.8200529217720032, -0...","[[0.44503530859947205, 0.8200529217720032, -0....","[-0.8344268798828125, -0.8343082070350647, -0....","[-0.8978692889213562, -0.9167898297309875, -0....","[17.381221771240234, 0.048995692282915115, 457...","[13.60135555267334, 0.035032469779253006, 6788...",1.060416,1.057065,17.381222,18.431328,18.373073
76,amg,minivite,"[[0.6035680770874023, 0.8200529217720032, -0.5...","[[0.10417995601892471, 0.8200529217720032, -0....","[-0.9116873145103455, -0.9033464789390564, 0.0...","[-0.26013436913490295, -0.3027505874633789, -0...","[12.778085708618164, 0.03730827942490578, 1064...","[51.597267150878906, 0.13898253440856934, 2408...",1.048583,1.084888,12.778086,13.398878,13.862786


In [ ]:
import matplotlib.pyplot as plt

def plot_times(df):
    for row in df.iloc:
        """Create a bar plot for a single row showing app_A_iso_time, app_A_y_true_time, app_A_y_pred_time"""
        times = [
            row['app_A_iso_time'],
            row['app_A_y_true_time'],
            row['app_A_y_pred_time']
        ]
        
        labels = ['Iso Time', 'Y True Time', 'Y Pred Time']
        apps = f"{row['app_A']} vs {row['app_B']}"
        
        fig, ax = plt.subplots(figsize=(8, 6))
        x = range(len(labels))
        bars = ax.bar(x, times, color=['#1f77b4', '#2ca02c', '#ff7f0e'])
        
        ax.set_ylabel('Time (seconds)')
        ax.set_title(f'Time Comparison: {apps}')
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.grid(axis='y', alpha=0.3)
        
        ax.bar_label(bars, padding=3)
        
        plt.tight_layout()
        plt.show()

# plot_times(test_df)
train_mse = mean_squared_error(train_df["app_A_y_true_time"], train_df["app_A_y_pred_time"])
test_mse = mean_squared_error(test_df["app_A_y_true_time"], test_df["app_A_y_pred_time"])
iso_mse = mean_squared_error(test_df["app_A_y_true_time"], test_df["app_A_iso_time"])

train_mae = mean_absolute_error(train_df["app_A_y_true_time"], train_df["app_A_y_pred_time"])
test_mae = mean_absolute_error(test_df["app_A_y_true_time"], test_df["app_A_y_pred_time"])
iso_mae = mean_absolute_error(test_df["app_A_y_true_time"], test_df["app_A_iso_time"])

train_mape = mean_absolute_percentage_error(train_df["app_A_y_true_time"], train_df["app_A_y_pred_time"]) * 100
test_mape = mean_absolute_percentage_error(test_df["app_A_y_true_time"], test_df["app_A_y_pred_time"]) * 100
iso_mape = mean_absolute_percentage_error(test_df["app_A_y_true_time"], test_df["app_A_iso_time"]) * 100

train_rmse = np.sqrt(train_mse) 
test_rmse = np.sqrt(test_mse) 
iso_rmse = np.sqrt(iso_mse) 

print("Train MSE:", train_mse)
print("Train RMSE:", train_rmse)
print("Train MAE:", train_mae)
print("Train MAPE:", train_mape, "%")
print(50*"-")
print("Test MSE:", test_mse)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test MAPE:", test_mape, "%")
# print(50*"-")
# print("Iso MSE:", iso_mse)
# print("Iso RMSE:", iso_rmse)
# print("Iso MAE:", iso_mae)
# print("Iso MAPE:", iso_mape, "%")

Train MSE: 28.999948097100177
Train RMSE: 5.385159988069081
Train MAE: 3.0342544965402247
Train MAPE: 6.122111782789165 %
--------------------------------------------------
Test MSE: 94.51486142472983
Test RMSE: 9.721875406768481
Test MAE: 5.98759293458124
Test MAPE: 8.339603323940885 %
--------------------------------------------------
Iso MSE: 230.7676166984196
Iso RMSE: 15.191037380587924
Iso MAE: 7.723725009944923
Iso MAPE: 9.207996948727597 %
